In [1]:
# Load the extension that allows us to compile CUDA code in python notebooks
# Documentation is here: https://nvcc4jupyter.readthedocs.io/en/latest/
!pip install git+https://github.com/andreinechaev/nvcc4jupyter.git
%load_ext nvcc4jupyter

  Cloning https://github.com/andreinechaev/nvcc4jupyter.git to /tmp/pip-req-build-zva8zjt7
  Running command git clone --filter=blob:none --quiet https://github.com/andreinechaev/nvcc4jupyter.git /tmp/pip-req-build-zva8zjt7
  Resolved https://github.com/andreinechaev/nvcc4jupyter.git to commit 173e636d9a3cb81c6101a160c1c8c86864eaaf5b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for nvcc4jupyter: filename=nvcc4jupyter-1.2.2-py3-none-any.whl size=12543 sha256=8f65034aebde6b8dc82156637db9fe8af44b7d617659c03fe9e6b1bca85ff8b4
  Stored in directory: /tmp/pip-ephem-wheel-cache-y0me9pe6/wheels/2f/1e/79/584a8d85ad018ecc510baa90d4fbc161bd56a4d66943d5b0d4
Successfully built nvcc4jupyter
Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmpnyrrd7c0".


In [2]:
%%cuda_group_save -g "source" -n "data_types.h"
/**
 * A collection of commonly used data types throughout this project.
 */
#pragma once

#include <stdint.h> // uint32_t

using element_t = uint32_t;

In [3]:
%%cuda_group_save -g "source" -n "cuda_common.h"
/**
 * Standard macros that can be useful for error checking.
 * https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__ERROR.html
 */
#pragma once

#include <cuda.h>

#define CUDA_CALL(exp)                                       \
    do {                                                     \
        cudaError res = (exp);                               \
        if(res != cudaSuccess) {                             \
            printf("Error at %s:%d\n %s\n",                  \
                __FILE__,__LINE__, cudaGetErrorString(res)); \
           exit(EXIT_FAILURE);                               \
        }                                                    \
    } while(0)

#define CHECK_ERROR(msg)                                             \
    do {                                                             \
        cudaError_t err = cudaGetLastError();                        \
        if(cudaSuccess != err) {                                     \
            printf("Error (%s) at %s:%d\n %s\n",                     \
                (msg), __FILE__, __LINE__, cudaGetErrorString(err)); \
            exit(EXIT_FAILURE);                                      \
        }                                                            \
    } while (0)

In [4]:
%%cuda_group_save -g "source" -n "data_generator.h"
/**
 * Functions for generating random input data with a fixed seed
 */
#pragma once

#include <random>  // for std::mt19937, std::uniform_int_distribution
#include <vector>
#include <algorithm>

#include "data_types.h"

namespace csc485b {
namespace a1 {

/**
 * Generates and returns a vector of random uniform data of a given length, n,
 * for any integral type. Input range will be [0, 2n].
 */
template < typename T >
std::vector< T > generate_uniform( std::size_t n )
{
    // for details of random number generation, see:
    // https://en.cppreference.com/w/cpp/numeric/random/uniform_int_distribution
    std::size_t random_seed = 20240916;  // use magic seed
    std::mt19937 rng( random_seed );     // use mersenne twister generator
    std::uniform_int_distribution<> distrib(0, 2 * n);

    std::vector< T > random_data( n ); // init array
    std::generate( std::begin( random_data )
                 , std::end  ( random_data )
                 , [ &rng, &distrib ](){ return static_cast< T >( distrib( rng ) ); });

    return random_data;
}

} // namespace a1
} // namespace csc485b

In [35]:
%%cuda_group_save -g "source" -n "algorithm_choices.h"
#pragma once

#include <vector>

#include "data_types.h"

namespace csc485b {
namespace a1 {
namespace cpu {

std::vector<element_t> run_cpu_baseline( std::vector< element_t > data, std::size_t switch_at, std::size_t n );

} // namespace cpu


namespace gpu {

struct GPUResults
{
    std::vector<element_t> opposing_result;
    std::vector<element_t> opposing_shared_result;
    std::vector<element_t> radix_result;
    std::vector<element_t> radix_shared_result;
};

GPUResults run_gpu_soln(std::vector<element_t> data, std::size_t switch_at, std::size_t n);

} // namespace gpu
} // namespace a1
} // namespace csc485b

In [24]:
%%cuda_group_save -g "source" -n "cpu_baseline.cu"
/**
 * CPU methods that the GPU should outperform.
 */

#include "algorithm_choices.h"

#include <algorithm> // std::sort()
#include <chrono>    // for timing
#include <iostream>  // std::cout, std::endl

namespace csc485b {
namespace a1      {
namespace cpu     {

/**
 * Simple solution that just sorts the whole array with a built-in sort
 * function and then resorts the last portion in the opposing order with
 * a second call to that same built-in sort function.
 */
void opposing_sort( element_t * data, std::size_t invert_at_pos, std::size_t num_elements )
{
    std::sort( data, data + num_elements, std::less< element_t >{} );
    std::sort( data + invert_at_pos, data + num_elements, std::greater< element_t >{} );
}

/**
 * Run the single-threaded CPU baseline that students are supposed to outperform
 * in order to obtain higher grades on this assignment. Times the execution and
 * prints to the standard output (e.g., the screen) that "wall time." Note that
 * the functions takes the input by value so as to not perturb the original data
 * in place.
 */
std::vector<element_t> run_cpu_baseline( std::vector< element_t > data, std::size_t switch_at, std::size_t n )
{
    auto const cpu_start = std::chrono::high_resolution_clock::now();
    opposing_sort( data.data(), switch_at, n );
    auto const cpu_end = std::chrono::high_resolution_clock::now();

    std::cout << "CPU Baseline time: "
              << std::chrono::duration_cast<std::chrono::nanoseconds>(cpu_end - cpu_start).count()
              << " ns" << std::endl;

    /*std::vector<element_t> last(data.end() - n/4, data.end());

    for( auto const x : last ) std::cout << x << " "; std::cout << std::endl;*/

    return data;
}

} // namespace cpu
} // namespace a1
} // namespace csc485b

In [36]:
%%cuda_group_save -g "source" -n "gpu_solution.cu"
/**
 * The file in which you will implement your GPU solutions!
 */

#include "algorithm_choices.h"

#include <chrono>    // for timing
#include <iostream>  // std::cout, std::endl

#include "cuda_common.h"

namespace csc485b {
namespace a1      {
namespace gpu     {

/**
 * The CPU baseline benefits from warm caches because the data was generated on
 * the CPU. Run the data through the GPU once with some arbitrary logic to
 * ensure that the GPU cache is warm too and the comparison is more fair.
 */
__global__
void warm_the_gpu( element_t * data, std::size_t invert_at_pos, std::size_t num_elements )
{
    int const th_id = blockIdx.x * blockDim.x + threadIdx.x;

    // We know this will never be true, because of the data generator logic,
    // but I doubt that the compiler will figure it out. Thus every element
    // should be read, but none of them should be modified.
    if( th_id < num_elements && data[ th_id ] > num_elements * 100 )
    {
        ++data[ th_id ]; // should not be possible.
    }
}

/**
 * Your solution. Should match the CPU output.
 */
__global__
void opposing_sort( element_t * data, std::size_t invert_at_pos, std::size_t num_elements )
{
    int const th_id = blockIdx.x * blockDim.x + threadIdx.x;

    if( th_id >= num_elements ) { return; }

    // Bitonic Sort

    for (std::size_t stride = 1; stride < num_elements ; stride <<= 1 )
    {
        for (std::size_t strd = stride; strd >= 1; strd >>= 1)
        {
            __syncthreads();
            if (th_id % (2 * strd) < strd)
            {
                if (th_id % (4 * stride) < 2 * stride)
                {
                    if (data[th_id] > data[th_id + strd])
                    {
                        int temp = data[th_id];
                        data[th_id] = data[th_id + strd];
                        data[th_id + strd] = temp;
                    }
                }
                else
                {
                    if (data[th_id] < data[th_id + strd])
                    {
                        int temp = data[th_id];
                        data[th_id] = data[th_id + strd];
                        data[th_id + strd] = temp;
                    }
                }
            }
        }
    }

    // [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
    // th_id = 3n/4 = 9
    // j = n - 1 + 3n/4 - th_id

    __syncthreads();

    if(th_id >= invert_at_pos){

      int last_half = (num_elements - invert_at_pos)/2;

      if (th_id < invert_at_pos + last_half){
        int j = num_elements - 1 - (th_id - invert_at_pos);

        int temp = data[th_id];
        data[th_id] = data[j];
        data[j] = temp;
      }
    }
}

// opposing sort using shared memory
 __global__
void opposing_sort_shared( element_t * data, std::size_t invert_at_pos, std::size_t num_elements )
{
    int const th_id = blockIdx.x * blockDim.x + threadIdx.x;

    if( th_id >= num_elements ) { return; }

    // Load into shared memory
    __shared__ element_t s[1024];
    s[th_id] = data[th_id];

    // Bitonic Sort

    for (std::size_t stride = 1; stride < num_elements ; stride <<= 1 )
    {
        for (std::size_t strd = stride; strd >= 1; strd >>= 1)
        {
            __syncthreads();
            if (th_id % (2 * strd) < strd)
            {
                if (th_id % (4 * stride) < 2 * stride)
                {
                    if (s[th_id] > s[th_id + strd])
                    {
                        int temp = s[th_id];
                        s[th_id] = s[th_id + strd];
                        s[th_id + strd] = temp;
                    }
                }
                else
                {
                    if (s[th_id] < s[th_id + strd])
                    {
                        int temp = s[th_id];
                        s[th_id] = s[th_id + strd];
                        s[th_id + strd] = temp;
                    }
                }
            }
        }
    }

    // [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
    // th_id = 3n/4 = 9
    // j = n - 1 + 3n/4 - th_id

    __syncthreads();

    if(th_id >= invert_at_pos){

      int last_half = (num_elements - invert_at_pos)/2;

      if (th_id < invert_at_pos + last_half){
        int j = num_elements - 1 - (th_id - invert_at_pos);

        int temp = s[th_id];
        s[th_id] = s[j];
        s[j] = temp;
      }
    }

    // Write back to global memory
    __syncthreads();
    data[th_id] = s[th_id];
}


__global__
void reverse_last(element_t * data, std::size_t invert_at_pos, std::size_t num_elements){
    int const th_id = blockIdx.x * blockDim.x + threadIdx.x;
    if( th_id >= num_elements ) { return; }

    if(th_id >= invert_at_pos){

      int last_half = (num_elements - invert_at_pos)/2;

      if (th_id < invert_at_pos + last_half){
        int j = num_elements - 1 - (th_id - invert_at_pos);

        int temp = data[th_id];
        data[th_id] = data[j];
        data[j] = temp;
      }
    }
}


__global__
void radix_sort_pass(const element_t* input, element_t* output, int n, int bit)
{
    int const th_id = blockIdx.x * blockDim.x + threadIdx.x;

    if (th_id >= n) { return; }

    element_t const my_key = input[th_id];

    int const my_bit = (my_key >> bit) & 1u;

    int total_zeros = 0;
    int matches_before = 0;

    for (int j = 0; j < n; j++)
    {
        int const other_bit = (input[j] >> bit) & 1u;

        if (other_bit == 0) total_zeros++;

        if (j < th_id && other_bit == my_bit) matches_before++;
    }

    int const bin_start = my_bit == 0 ? 0 : total_zeros;

    int const new_index = bin_start + matches_before;

    output[new_index] = my_key;
}

void run_radix_sort(element_t* d_data, std::size_t invert_at_pos, int n)
{
    if (n <= 0) return;

    element_t* d_temp;
    cudaMalloc((void**)&d_temp, n * sizeof(element_t));

    element_t* input = d_data;
    element_t* output = d_temp;

    for (int bit = 0; bit < 12; bit++)
    {
        radix_sort_pass<<<1, 1024>>>(input, output, n, bit);

        std::swap(input, output);
    }

    reverse_last<<<1, 1024>>>(input, invert_at_pos, n);

    if (input != d_data)
    {
        cudaMemcpy(d_data, input, n * sizeof(element_t), cudaMemcpyDeviceToDevice);
    }

    cudaFree(d_temp);
}


// Radix sort using shared memory
__global__
void radix_sort_shared( element_t * data, std::size_t invert_at_pos, std::size_t num_elements){
    int const th_id = blockIdx.x * blockDim.x + threadIdx.x;
    if( th_id >= num_elements ) { return; }

    int const lane_id = threadIdx.x % 32;
    int const warp_id = threadIdx.x / 32;

    // For 1024 elements in one block
    __shared__ element_t a[1024];
    __shared__ element_t b[1024];
    __shared__ int warp_zeros[32];
    __shared__ int zeros_before_warp[32];
    __shared__ int total_zeros;

    a[th_id] = data[th_id];
    __syncthreads();

    element_t* input = a;
    element_t* output = b;

    // Sort one bit at a time
    for (int bit = 0; bit < 12; bit++)
    {
        element_t const my_key = input[th_id];
        int const my_bit = (my_key >> bit) & 1u;

        uint32_t const zero_bits = __ballot_sync(0xFFFFFFFFU, my_bit == 0);

        uint32_t const match_me = __match_any_sync(0xFFFFFFFFU, my_bit);

        uint32_t const mask_prev_threads = (1u << lane_id) - 1u;

        int const matches_before = __popc(match_me & mask_prev_threads);

        // Each warp counts its zeros
        if (lane_id == 0) warp_zeros[warp_id] = __popc(zero_bits);

        __syncthreads();

        if (th_id == 0)
        {
            int sum = 0;

            for (int w = 0; w < 32; w++)
            {
                zeros_before_warp[w] = sum;
                sum += warp_zeros[w];
            }

            total_zeros = sum;
        }

        __syncthreads();

        int zeros_before = zeros_before_warp[warp_id];

        // Calculate destination
        int bin_start;

        if (my_bit == 0)
            bin_start = zeros_before;
        else
            bin_start = total_zeros
                      + warp_id * 32
                      - zeros_before;

        int const new_index = bin_start + matches_before;

        output[new_index] = my_key;

        __syncthreads();

        // Swap buffers for next pass
        element_t* temp = input;
        input = output;
        output = temp;
    }

    // Copy sorted elements back
    data[th_id] = input[th_id];

    __syncthreads();

    if(th_id >= invert_at_pos){

      int last_half = (num_elements - invert_at_pos)/2;

      if (th_id < invert_at_pos + last_half){
        int j = num_elements - 1 - (th_id - invert_at_pos);

        int temp = data[th_id];
        data[th_id] = data[j];
        data[j] = temp;
      }
    }
}


/**
 * Performs all the logic of allocating device vectors and copying host/input
 * vectors to the device. Times the opposing_sort() kernel with wall time,
 * but excludes set up and tear down costs such as mallocs, frees, and memcpies.
 */
GPUResults run_gpu_soln( std::vector< element_t > data, std::size_t switch_at, std::size_t n )
{
    // Kernel launch configurations. Feel free to change these.
    // This is set to maximise the size of a thread block on a T4, but it hasn't
    // been tuned. It's not known if this is optimal.
    std::size_t const threads_per_block = 1024;
    std::size_t const num_blocks =  ( n + threads_per_block - 1 ) / threads_per_block;

    // Number of timed runs to average over
    int const runs = 100;

    // Allocate arrays on the device/GPU
    element_t * d_data;
    cudaMalloc( (void**) & d_data, sizeof( element_t ) * n );
    CHECK_ERROR("Allocating input array on device");

    // Copy the input from the host to the device/GPU
    cudaMemcpy( d_data, data.data(), sizeof( element_t ) * n, cudaMemcpyHostToDevice );
    CHECK_ERROR("Copying input array to device");

    // Warm the cache on the GPU for a more fair comparison
    warm_the_gpu<<< num_blocks, threads_per_block>>>( d_data, switch_at, n );
    cudaDeviceSynchronize();

    // ----- Opposing sort (global memory) -----

    long long total = 0;

    for (int r = 0; r <= runs; ++r)
    {
        // Fresh unsorted input each run (not timed)
        cudaMemcpy( d_data, data.data(), sizeof( element_t ) * n, cudaMemcpyHostToDevice );
        CHECK_ERROR("Copying input array to device");

        auto const kernel_start = std::chrono::high_resolution_clock::now();
        opposing_sort<<< num_blocks, threads_per_block>>>( d_data, switch_at, n );
        cudaDeviceSynchronize();
        auto const kernel_end = std::chrono::high_resolution_clock::now();
        CHECK_ERROR("Executing kernel on device");

        // Skip run 0: it includes the one-time first-launch cost
        if (r > 0)
            total += std::chrono::duration_cast<std::chrono::nanoseconds>(kernel_end - kernel_start).count();
    }

    std::cout << "GPU Solution time (Opposing), avg of " << runs << " runs: "
              << total / runs << " ns" << std::endl;

    std::vector<element_t> opposing_result(n);

    cudaMemcpy(opposing_result.data(), d_data, sizeof(element_t) * n, cudaMemcpyDeviceToHost);
    CHECK_ERROR("Copying opposing sort result");

    /*std::vector<element_t> last( opposing_result.end() - n/4, opposing_result.end());

    for( auto const x : last ) std::cout << x << " "; std::cout << std::endl;*/

    // ----- Opposing sort with shared memory -----

    long long total_shared = 0;

    for (int r = 0; r <= runs; ++r)
    {
        // Fresh unsorted input each run (not timed)
        cudaMemcpy( d_data, data.data(), sizeof( element_t ) * n, cudaMemcpyHostToDevice );
        CHECK_ERROR("Copying input array to device");

        auto const kernel_start = std::chrono::high_resolution_clock::now();
        opposing_sort_shared<<< num_blocks, threads_per_block>>>( d_data, switch_at, n );
        cudaDeviceSynchronize();
        auto const kernel_end = std::chrono::high_resolution_clock::now();
        CHECK_ERROR("Executing shared opposing sort on device");

        // Skip run 0: it includes the one-time first-launch cost
        if (r > 0)
            total_shared += std::chrono::duration_cast<std::chrono::nanoseconds>(kernel_end - kernel_start).count();
    }

    std::cout << "GPU Solution time (Opposing, shared), avg of " << runs << " runs: "
              << total_shared / runs << " ns" << std::endl;

    std::vector<element_t> opposing_shared_result(n);

    cudaMemcpy(opposing_shared_result.data(), d_data, sizeof(element_t) * n, cudaMemcpyDeviceToHost);
    CHECK_ERROR("Copying shared opposing sort result");

    /*last.assign(opposing_shared_result.end() - n/4, opposing_shared_result.end());

    for( auto const x : last ) std::cout << x << " "; std::cout << std::endl;

    std::cout << "Opposing (global) matches Opposing (shared): " << std::boolalpha
              << (opposing_result == opposing_shared_result) << std::endl;*/


    // ----- Radix sort (global memory) -----

    long long total_radix = 0;

    for (int r = 0; r <= runs; ++r)
    {
        // Fresh unsorted input each run (not timed)
        cudaMemcpy(d_data, data.data(), sizeof(element_t) * n,
                  cudaMemcpyHostToDevice);
        CHECK_ERROR("Copying input array to device");

        auto const kernel_start = std::chrono::high_resolution_clock::now();

        run_radix_sort(d_data, switch_at, n);

        cudaDeviceSynchronize();

        auto const kernel_end = std::chrono::high_resolution_clock::now();
        CHECK_ERROR("Executing global radix sort");

        if (r > 0)
            total_radix += std::chrono::duration_cast<std::chrono::nanoseconds>(
                kernel_end - kernel_start).count();
    }

    std::cout << "GPU Solution time (Radix, global), avg of "
              << runs << " runs: "
              << total_radix / runs << " ns" << std::endl;

    std::vector<element_t> radix_result(n);

    cudaMemcpy(radix_result.data(), d_data, sizeof(element_t) * n,
              cudaMemcpyDeviceToHost);
    CHECK_ERROR("Copying global radix result");

    /*last.assign(radix_result.end() - n/4, radix_result.end());

    for (auto const x : last) std::cout << x << " ";
    std::cout << std::endl;*/


    // ----- Radix sort (shared memory) -----

    long long total_radix_shared = 0;

    for (int r = 0; r <= runs; ++r)
    {
        // Fresh unsorted input each run (not timed)
        cudaMemcpy(d_data, data.data(), sizeof(element_t) * n,
                  cudaMemcpyHostToDevice);
        CHECK_ERROR("Copying input array to device");

        auto const kernel_start = std::chrono::high_resolution_clock::now();

        radix_sort_shared<<<num_blocks, threads_per_block>>>(
            d_data, switch_at, n
        );

        cudaDeviceSynchronize();

        auto const kernel_end = std::chrono::high_resolution_clock::now();
        CHECK_ERROR("Executing shared radix sort");

        if (r > 0)
            total_radix_shared += std::chrono::duration_cast<std::chrono::nanoseconds>(
                kernel_end - kernel_start).count();
    }

    std::cout << "GPU Solution time (Radix, shared), avg of "
              << runs << " runs: "
              << total_radix_shared / runs << " ns" << std::endl;

    std::vector<element_t> radix_shared_result(n);

    cudaMemcpy(radix_shared_result.data(), d_data, sizeof(element_t) * n,
              cudaMemcpyDeviceToHost);
    CHECK_ERROR("Copying shared radix result");

    /*last.assign(radix_shared_result.end() - n/4, radix_shared_result.end());

    for (auto const x : last) std::cout << x << " ";
    std::cout << std::endl;*/

    cudaFree( d_data );
    CHECK_ERROR("Freeing device memory");

    return {
        std::move(opposing_result),
        std::move(opposing_shared_result),
        std::move(radix_result),
        std::move(radix_shared_result)
    };
}

} // namespace gpu
} // namespace a1
} // namespace csc485b

In [41]:
%%cuda_group_save -g "source" -n "main.cu"
/**
 * Driver for the benchmark comparison. Generates random data,
 * runs the CPU baseline, and then runs your code.
 */

#include <cstddef>  // std::size_t type
#include <iostream> // std::cout, std::endl
#include <vector>

#include "algorithm_choices.h"
#include "data_generator.h"
#include "data_types.h"
#include "cuda_common.h"

int main()
{
    std::size_t const n = 1024;
    std::size_t const switch_at = 3 * ( n >> 2 ) ;

    auto data = csc485b::a1::generate_uniform< element_t >( n );
    // for( auto const x : data ) std::cout << x << " "; std::cout << std::endl;
    std::vector<element_t> cpu_result = csc485b::a1::cpu::run_cpu_baseline( data, switch_at, n );
    auto gpu_results = csc485b::a1::gpu::run_gpu_soln( data, switch_at, n );

    // ----- Correctness checks -----

    bool opposing_correct =
        gpu_results.opposing_result == cpu_result;

    bool opposing_shared_correct =
        gpu_results.opposing_shared_result == cpu_result;

    bool radix_correct =
        gpu_results.radix_result == cpu_result;

    bool radix_shared_correct =
        gpu_results.radix_shared_result == cpu_result;

    std::cout << "\n----- Correctness Results -----\n";

    std::cout << std::boolalpha;

    std::cout << "Opposing (global): "
              << opposing_correct << '\n';

    std::cout << "Opposing (shared): "
              << opposing_shared_correct << '\n';

    std::cout << "Radix (global): "
              << radix_correct << '\n';

    std::cout << "Radix (shared): "
              << radix_shared_correct << '\n';

    // ----- Show first mismatch for each algorithm -----

    auto check_mismatch = [&](const char* name,
                              const std::vector<element_t>& result)
    {
        if (result == cpu_result)
            return;

        for (std::size_t i = 0; i < n; i++)
        {
            if (cpu_result[i] != result[i])
            {
                std::cout << name
                          << " mismatch at index " << i
                          << "\nCPU: " << cpu_result[i]
                          << "\nGPU: " << result[i]
                          << '\n';
                break;
            }
        }
    };

    check_mismatch("Opposing global", gpu_results.opposing_result);
    check_mismatch("Opposing shared", gpu_results.opposing_shared_result);
    check_mismatch("Radix global", gpu_results.radix_result);
    check_mismatch("Radix shared", gpu_results.radix_shared_result);

    // ----- Final result -----

    bool all_correct =
        opposing_correct &&
        opposing_shared_correct &&
        radix_correct &&
        radix_shared_correct;

    std::cout << "\nAll results match CPU: "
              << all_correct << std::endl;

    return all_correct ? EXIT_SUCCESS : EXIT_FAILURE;
}

In [39]:
%cuda_group_run --group "source" --compiler-args "-O0 -g -std=c++20 -arch=sm_75"

CPU Baseline time: 206745 ns
GPU Solution time (Opposing), avg of 100 runs: 92492 ns
GPU Solution time (Opposing, shared), avg of 100 runs: 71035 ns
GPU Solution time (Radix, global), avg of 100 runs: 1548796 ns
GPU Solution time (Radix, shared), avg of 100 runs: 18636 ns

----- Correctness Results -----
Opposing (global): true
Opposing (shared): true
Radix (global): true
Radix (shared): true

All results match CPU: true



In [40]:
%cuda_group_run --group "source" --compiler-args "-O3 -g -std=c++20 -arch=sm_75"

CPU Baseline time: 43751 ns
GPU Solution time (Opposing), avg of 100 runs: 114280 ns
GPU Solution time (Opposing, shared), avg of 100 runs: 106266 ns
GPU Solution time (Radix, global), avg of 100 runs: 2473026 ns
GPU Solution time (Radix, shared), avg of 100 runs: 18264 ns

----- Correctness Results -----
Opposing (global): true
Opposing (shared): true
Radix (global): true
Radix (shared): true

All results match CPU: true

